<a href="https://colab.research.google.com/github/ritesh23r/insurance-cost-prediction-data/blob/main/Insurance_Cost_Prediction_EDA%2CFeature_Engineering_%26_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv('insurance.csv')

In [ ]:
data

# **EDA**

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
data.isnull().sum()

In [ ]:
data.columns

In [ ]:
numeric_col = ['age', 'bmi', 'children', 'charges']
for col in numeric_col :
  plt.figure(figsize=(6,4))
  sns.histplot(data[col], kde = False , bins = 20)

In [ ]:
sns.countplot(x=data['sex'])

In [ ]:
sns.countplot(x=data['smoker'])

In [ ]:
sns.countplot(x=data['children'])

In [ ]:
for col in numeric_col :
  plt.figure(figsize=(6,4))
  sns.boxplot(data[col])

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(data.corr(numeric_only=True), annot = True)

# **Data Cleaning and Preprocessing**

In [ ]:
data_cleaned = data.copy()

In [ ]:
data_cleaned.drop_duplicates(inplace = True)

In [ ]:
data_cleaned.shape

In [ ]:
data_cleaned.isnull().sum()

In [ ]:
data_cleaned['sex'].value_counts()

In [ ]:
data_cleaned['sex'] = data_cleaned['sex'].map({'male':0, 'female':1})

In [ ]:
data_cleaned['smoker'] = data_cleaned['smoker'].map({'yes':1 , 'no' : 0})

In [ ]:
data_cleaned.rename(
  columns ={
      'sex' : 'is_female',
      'smoker' : 'is_smoker'
  },inplace = True
)

In [ ]:
data_cleaned['region'].value_counts()

In [ ]:
data_cleaned = pd.get_dummies(data_cleaned, columns=['region'],drop_first=True)

In [ ]:
data_cleaned=data_cleaned.astype(int)

# **Feature Engineering and Extraction**

In [ ]:
sns.histplot(data_cleaned['bmi'])

In [ ]:
data_cleaned['bmi_category'] = pd.cut(
    data_cleaned['bmi'],
    bins=[0, 18.5, 24.9, 29.9, float('inf')],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)

In [ ]:
data_cleaned = pd.get_dummies(data_cleaned,columns=['bmi_category'],drop_first=True)

In [ ]:
data_cleaned=data_cleaned.astype(int)

In [ ]:
from sklearn.preprocessing import StandardScaler
cols = ['age','bmi','children']
scaler = StandardScaler()
data_cleaned[cols] = scaler.fit_transform(data_cleaned[cols])

In [ ]:
data_cleaned.head()

In [ ]:
data_cleaned.corr()['charges'].sort_values(ascending=False)

In [ ]:
cat_features = [
    'is_female', 'is_smoker',
    'region_northwest', 'region_southeast', 'region_southwest',
    'bmi_category_Normal', 'bmi_category_Overweight', 'bmi_category_Obese'
]

In [ ]:
from scipy.stats import chi2_contingency
import pandas as pd

alpha = 0.05

data_cleaned['charges_bin'] = pd.qcut(data_cleaned['charges'], q=4, labels=False)
chi2_results = {}

for col in cat_features:
    # Corrected df_cleaned to data_cleaned
    contingency = pd.crosstab(data_cleaned[col], data_cleaned['charges_bin'])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency)
    decision = 'Reject Null (Keep Feature)' if p_val < alpha else 'Accept Null (Drop Feature)'
    chi2_results[col] = {
        'chi2_statistic': chi2_stat,
        'p_value': p_val,
        'Decision': decision
    }

chi2_df = pd.DataFrame(chi2_results).T
chi2_df = chi2_df.sort_values(by='p_value')
chi2_df

In [ ]:
final_data = data_cleaned[['age', 'is_female', 'bmi', 'children', 'is_smoker', 'region_southeast','charges','bmi_category_Obese']]

In [ ]:
final_data